In [ ]:
import json
import os
import os.path as osp
from typing import List, Tuple
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import accuracy_score, f1_score
from uncertainty_metrics import compute_uncertainty_metrics


def _logsumexp(arr: np.ndarray, axis: int = 1, keepdims: bool = True) -> np.ndarray:
    """Compute numerically stable log-sum-exp.

    Args:
        arr (np.ndarray): Input array.
        axis (int): Axis over which to reduce.
        keepdims (bool): Whether to keep reduced dimensions.

    Returns:
        np.ndarray: Log-sum-exp result.
    """
    m = np.max(arr, axis=axis, keepdims=True)
    return m + np.log(np.sum(np.exp(arr - m), axis=axis, keepdims=True) + 1e-12)

def _weighted_nll(weights: np.ndarray, p_base: np.ndarray, p_sidekick: np.ndarray, y: np.ndarray) -> float:
    """Negative log-likelihood for a weighted log-probability ensemble.

    Args:
        weights (np.ndarray): Base and sidekick weights.
        p_base (np.ndarray): Base probabilities.
        p_sidekick (np.ndarray): Sidekick probabilities.
        y (np.ndarray): True labels.

    Returns:
        float: Mean negative log-likelihood.
    """
    weight_base, weight_sidekick = weights
    log_p = weight_base * (p_base + 1e-12) + weight_sidekick * (p_sidekick + 1e-12)
    #log_p = weight_base * np.log(p_base + 1e-12) + weight_sidekick * np.log(p_sidekick + 1e-12)
    log_p_norm = log_p - _logsumexp(log_p, axis=1, keepdims=True)
    return -np.mean(log_p_norm[np.arange(len(y)), y])

def _softmax_from_log(log_p: np.ndarray, ax: int = 1) -> np.ndarray:
    """Convert log-probabilities to normalized probabilities.

    Args:
        log_p (np.ndarray): Log-probability matrix.

    Returns:
        np.ndarray: Normalized probability matrix.
    """
    log_p_norm = log_p - _logsumexp(log_p, axis=ax, keepdims=True)
    return np.exp(log_p_norm)

def _combine_probs(w_base: float, w_sidekick: float, logits_a: np.ndarray, logits_b: np.ndarray) -> np.ndarray:
    """Combine two probability matrices using log-space weighting.

    Args:
        w (float): Weight for the first probability matrix.
        logits_a (np.ndarray): First logit matrix.
        logits_b (np.ndarray): Second logit matrix.

    Returns:
        np.ndarray: Combined probability matrix.
    """
    log_p = w_base * logits_a + w_sidekick * logits_b

    return _softmax_from_log(log_p)

def _build_metrics(split_name: str, w_base: float, w_sidekick: float, logits_a: np.ndarray, logits_b: np.ndarray, y_true: np.ndarray, ax:int = 1):
    """Compute base/sidekick/duo metrics for a split.

    Args:
        split_name (str): Split label used for logging.
        w (float): Weight for base probabilities.
        probs_a (np.ndarray): Base probabilities.
        probs_b (np.ndarray): Sidekick probabilities.
        y_true (np.ndarray): True label indices.

    Returns:
        tuple: (metrics dict, duo_probs, duo_pred).
    """

    probs_a = _softmax_from_log(logits_a)
    probs_b = _softmax_from_log(logits_b)
    base_pred = probs_a.argmax(axis=ax)
    sidekick_pred = probs_b.argmax(axis=ax)

    duo_probs = _combine_probs(w_base, w_sidekick, logits_a, logits_b)
    duo_pred = duo_probs.argmax(axis=ax)

    
    metrics = {
        "base_accuracy": accuracy_score(y_true, base_pred),
        "sidekick_accuracy": accuracy_score(y_true, sidekick_pred),
        "duo_accuracy": accuracy_score(y_true, duo_pred),
        "base_f1_macro": f1_score(y_true, base_pred, average="macro"),
        "sidekick_f1_macro": f1_score(y_true, sidekick_pred, average="macro"),
        "duo_f1_macro": f1_score(y_true, duo_pred, average="macro"),
    }
    

    metrics.update({f"base_{k}": v for k, v in compute_uncertainty_metrics(probs_a, y_true).items()})
    metrics.update({f"sidekick_{k}": v for k, v in compute_uncertainty_metrics(probs_b, y_true).items()})
    metrics.update({f"duo_{k}": v for k, v in compute_uncertainty_metrics(duo_probs, y_true).items()})
    return metrics, duo_probs, duo_pred

def _sort_by_idx(arr: np.ndarray, idx: np.ndarray) -> np.ndarray:
    """
    Reorder `arr` using indices `idx`.

    Supports:
      - arr shape: (N,)
      - arr shape: (B, N) or (B, N, ...)
      - idx shape: (N,) or (B, N)

    Returns:
      - reordered array with same shape as `arr`
    """
    arr = np.asarray(arr)
    idx = np.asarray(idx)

    if arr.ndim == 1:
        if idx.shape[0] != arr.shape[0]:
            raise ValueError(f"`idx` shape {idx.shape} incompatible with arr size {arr.shape[0]}")
        return arr[idx]
    elif arr.ndim >= 2:
        B, N = arr.shape[0], arr.shape[1]
        if idx.shape[0] != B:
            raise ValueError(f"`idx` shape {idx.shape} incompatible with arr axis=1 size {N}")
        return arr[idx]

    else:
        raise ValueError(f"`arr` must be at least 1D (got shape {arr.shape})")



In [71]:
# Directory for validation and test results
result_dir = '/Users/alicansahin/Desktop'
base_model_validation_results_path = osp.join(result_dir, f'base/val_preds', 'obqa.npz')
sidekick_model_validation_results_path = osp.join(result_dir, f'sidekick/val_preds', 'obqa.npz')
base_model_test_results_path = osp.join(result_dir, f'base/test_preds', 'obqa.npz')
sidekick_model_test_results_path = osp.join(result_dir, f'sidekick/test_preds', 'obqa.npz')


# Load validation and test results
base_val_results = dict(np.load(base_model_validation_results_path, allow_pickle = True))
sidekick_val_results = dict(np.load(sidekick_model_validation_results_path, allow_pickle = True))
base_test_results = dict(np.load(base_model_test_results_path, allow_pickle = True))
sidekick_test_results = dict(np.load(sidekick_model_test_results_path, allow_pickle = True))


In [72]:
# Filter data wrt seed value
base_val_results = base_val_results[str(42)].item()
sidekick_val_results = sidekick_val_results[str(42)].item()
base_test_results = base_test_results[str(42)].item()
sidekick_test_results = sidekick_test_results[str(42)].item()

# Ensure the indices match and sort accordingly
base_idx_order_val = np.argsort(base_val_results['idx'])
sidekick_idx_order_val = np.argsort(sidekick_val_results['idx'])
base_idx_order_test = np.argsort(base_test_results['idx'])
sidekick_idx_order_test = np.argsort(sidekick_test_results['idx'])


# Check if indices match
if not np.array_equal(_sort_by_idx(base_val_results['idx'], base_idx_order_val), _sort_by_idx(sidekick_val_results['idx'], sidekick_idx_order_val)):
    raise ValueError('The indices of the base and sidekick validation results do not match. Optimizer will not be reliable. Check the data')

if not np.array_equal(_sort_by_idx(base_test_results['idx'], base_idx_order_test), _sort_by_idx(sidekick_test_results['idx'], sidekick_idx_order_test)):
    raise ValueError('The indices of the base and sidekick test results do not match. Optimizer will not be reliable. Check the data')
    
# Sort logits based on indices
base_logits_val_sorted = _sort_by_idx(base_val_results['logits'], base_idx_order_val)
sidekick_logits_val_sorted = _sort_by_idx(sidekick_val_results['logits'], sidekick_idx_order_val)
base_logits_test_sorted = _sort_by_idx(base_test_results['logits'], base_idx_order_test)
sidekick_logits_test_sorted = _sort_by_idx(sidekick_test_results['logits'], sidekick_idx_order_test)


In [73]:
def __sanity_check_on_probs(probs, which_prob):
    total_observations = probs.shape[0]

    if int(probs.sum(axis=1).sum()) != total_observations:
        raise ValueError(f'Something wrong with {which_prob} probabilities. Check the logits and softmax function')

In [74]:
# Get validation and test probs and true labels
base_probs_val = _softmax_from_log(base_logits_val_sorted)
sidekick_probs_val = _softmax_from_log(sidekick_logits_val_sorted)
base_probs_test = _softmax_from_log(base_logits_test_sorted)
sidekick_probs_test = _softmax_from_log(sidekick_logits_test_sorted)

# Check the probs 
__sanity_check_on_probs(base_probs_val, 'base_logits_val')
__sanity_check_on_probs(sidekick_probs_val, 'sidekick_logits_val')
__sanity_check_on_probs(base_probs_test, 'base_logits_test')
__sanity_check_on_probs(sidekick_probs_test, 'sidekick_logits_test')

In [75]:
true_labels_val = _sort_by_idx(base_val_results['true_labels'], base_idx_order_val)
true_labels_test = _sort_by_idx(base_test_results['true_labels'], base_idx_order_test)

In [76]:
constrain_weights = True
verbose = True
optimizer_method = "SLSQP"
cons = {"type": "eq", "fun": lambda w: w[0] + w[1] - 1} if constrain_weights else ()
bounds = [(0.0, 1.0), (0.0, 1.0)] if constrain_weights else None
x0 = np.array([0.5, 0.5]) if constrain_weights else np.array([1.0, 1.0])

callback = None
if verbose:
    def _print_weights(xk):
        print(f"iter weights -> base: {xk[0]:.4f}, sidekick: {xk[1]:.4f}")
    callback = _print_weights

res = minimize(
    _weighted_nll,
    x0 = x0,
    args=(base_logits_val_sorted, sidekick_logits_val_sorted, true_labels_val),
    method = optimizer_method,
    bounds = bounds,
    constraints = cons,
    callback = callback,
    options = {'disp': verbose}

)

if not res.success:
    raise RuntimeError(f'Weight optimization failed: {res.message}')

w_base, w_sidekick = res.x

# Get the duo metrics for validation and test sets
print(f"Optimized weights -> base: {w_base:.4f}, sidekick: {w_sidekick:.4f}")



iter weights -> base: 0.3362, sidekick: 0.6638
iter weights -> base: 0.4413, sidekick: 0.5587
iter weights -> base: 0.4429, sidekick: 0.5571
iter weights -> base: 0.4429, sidekick: 0.5571
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.5214613852624446
            Iterations: 4
            Function evaluations: 13
            Gradient evaluations: 4
Optimized weights -> base: 0.4429, sidekick: 0.5571


In [79]:
val_metrics, duo_val_probs, duo_val_pred = _build_metrics("val", w_base, w_sidekick, base_logits_val_sorted, sidekick_logits_val_sorted, true_labels_val)
test_metrics, duo_test_probs, duo_test_pred = _build_metrics("test", w_base, w_sidekick, base_logits_test_sorted, sidekick_logits_test_sorted, true_labels_test)



/Users/alicansahin/Desktop/LMU/SoSe 26/consulting/asymmetric-llms/ib_edl/utils/uncertainty_metrics.py:48: RuntimeWarning: divide by zero encountered in log
  nll = -np.mean(np.log(true_probs + 1e-12))
/Users/alicansahin/Desktop/LMU/SoSe 26/consulting/asymmetric-llms/ib_edl/utils/uncertainty_metrics.py:49: RuntimeWarning: divide by zero encountered in log
  lppd = np.sum(np.log(true_probs + 1e-12))
/Users/alicansahin/Desktop/LMU/SoSe 26/consulting/asymmetric-llms/ib_edl/utils/uncertainty_metrics.py:48: RuntimeWarning: divide by zero encountered in log
  nll = -np.mean(np.log(true_probs + 1e-12))
/Users/alicansahin/Desktop/LMU/SoSe 26/consulting/asymmetric-llms/ib_edl/utils/uncertainty_metrics.py:49: RuntimeWarning: divide by zero encountered in log
  lppd = np.sum(np.log(true_probs + 1e-12))


In [80]:
val_metrics

{'base_accuracy': 0.888,
 'sidekick_accuracy': 0.822,
 'duo_accuracy': 0.894,
 'base_f1_macro': 0.886873358825446,
 'sidekick_f1_macro': 0.8207572780599983,
 'duo_f1_macro': 0.8928956535309299,
 'base_nll': inf,
 'base_lppd': -inf,
 'base_brier': 0.211669921875,
 'base_ece': 0.1005068359375,
 'base_mean_uncertainty': 0.01123046875,
 'sidekick_nll': 1.1435546875,
 'sidekick_lppd': -572.0,
 'sidekick_brier': 0.334716796875,
 'sidekick_ece': 0.16054248046875005,
 'sidekick_mean_uncertainty': 0.02099609375,
 'duo_nll': 0.5214613845321858,
 'duo_lppd': -260.7306922660929,
 'duo_brier': 0.17731987793114165,
 'duo_ece': 0.07636928705383525,
 'duo_mean_uncertainty': 0.0357212293251743}

In [81]:
test_metrics

{'base_accuracy': 0.884,
 'sidekick_accuracy': 0.834,
 'duo_accuracy': 0.892,
 'base_f1_macro': 0.8829031399332185,
 'sidekick_f1_macro': 0.8322125950561098,
 'duo_f1_macro': 0.8916974824112107,
 'base_nll': inf,
 'base_lppd': -inf,
 'base_brier': 0.2200927734375,
 'base_ece': 0.10668261718749997,
 'base_mean_uncertainty': 0.01171875,
 'sidekick_nll': 1.1083984375,
 'sidekick_lppd': -554.5,
 'sidekick_brier': 0.31787109375,
 'sidekick_ece': 0.15281933593750005,
 'sidekick_mean_uncertainty': 0.0205078125,
 'duo_nll': 0.5901198557872067,
 'duo_lppd': -295.05992789360334,
 'duo_brier': 0.1848759160658537,
 'duo_ece': 0.07821689758980664,
 'duo_mean_uncertainty': 0.034021818794397585}